# Module 1 — Emotion vector extraction (Llama-3.1-8B-Instruct)

**v2 §3 Methodology, Llama parallel track.** Brayden-owned as of the 2026-05-29 module reassignment (M4 → teammate, Llama M1-M3 → Brayden). Pipeline mirrors `m1_extract.ipynb` but loads Llama instead of Gemma and writes to a separate `llama_L{layer}` namespace so Gemma artifacts on disk and HF are untouched.

**Pipeline:**
1. Stories are committed in `data/stories/{emotion}/*.txt` (shared with the Gemma run).
2. Run each story through Llama-3.1-8B-Instruct; capture residual-stream activations at layer 21 (~2/3 of 32 layers; working default — see config.yaml comment).
3. Average from token 50 onward → per-story vector.
4. Per-emotion mean across stories.
5. Subtract cross-emotion mean.
6. Project out top PCs of neutral corpus (50% variance).
7. ℓ₂-normalize.

**Run on:** A100 40GB (Llama-3.1-8B in bf16 ≈ 16GB). Fits on Pro+ A100 or any Vast/Lambda A100. ~2–3 GPU-hr.

**Prerequisite:** `sanity_test.ipynb` must pass first (validates the env).

## Cell 1 — env setup (Colab + SageMaker + local)

Detects runtime, pulls secrets the right way, and ensures we're inside the repo. On SageMaker, this expects `HF_TOKEN` to already be in env (set with `export HF_TOKEN=hf_...` in the terminal before launching Jupyter) and that you opened this notebook from inside the cloned `Algoverse/` directory.

In [ ]:
import os
import sys
import subprocess

# detect runtime
IS_COLAB = 'google.colab' in sys.modules
IS_SAGEMAKER = os.path.exists('/home/ec2-user/SageMaker') or 'SageMaker' in os.environ.get('PWD', '')
print(f'runtime: colab={IS_COLAB}, sagemaker={IS_SAGEMAKER}')

# secrets — Colab uses userdata; SageMaker/local expects them already in env.
# Two-token mode (optional): HF_MODEL_TOKEN authenticates the gated model download
# (e.g. a teammate's token who accepted the Llama license); HF_TOKEN authenticates
# Hub artifact-repo operations (your own token, with Write on the dataset).
# If HF_MODEL_TOKEN is unset, model_load falls back to HF_TOKEN for both.
if IS_COLAB:
	from google.colab import userdata
	os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
	try:
		os.environ['HF_MODEL_TOKEN'] = userdata.get('HF_MODEL_TOKEN')
	except Exception:
		pass  # single-token mode; model_load will use HF_TOKEN
else:
	assert 'HF_TOKEN' in os.environ, (
		'HF_TOKEN not set. Set it in the terminal (or at the top of this notebook):\n'
		'  export HF_TOKEN=hf_...   # token with Write on the artifact dataset repo\n'
		'Two-token mode (optional): also set HF_MODEL_TOKEN for the gated model download\n'
		'if a different account accepted the Llama license:\n'
		'  export HF_MODEL_TOKEN=hf_...\n'
		'Accept the model license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct'
	)

# repo: Colab clones fresh each session; SageMaker/local expects you started in the repo
if IS_COLAB:
	subprocess.run('git clone https://github.com/BraydenFeng/Algoverse.git || (cd Algoverse && git pull)', shell=True, check=True)
	os.chdir('Algoverse')
else:
	# Jupyter starts the kernel in the notebook's dir (notebooks/); walk up to repo root.
	if os.path.basename(os.getcwd()) == 'notebooks':
		os.chdir('..')
	assert os.path.isdir('src') and os.path.isfile('config.yaml'), (
		f'expected to be inside the Algoverse repo root, got cwd={os.getcwd()}. '
		'On SageMaker: `cd ~/SageMaker/Algoverse` and re-open this notebook from there.'
	)

# pip install is idempotent — fine to re-run on each session
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'cwd: {os.getcwd()}')


## Cell 1.5 — pre-flight: GPU, disk, dep check

In [ ]:
# pre-flight: GPU, disk, deps. Cheap — run before the model load to catch problems early.
import subprocess
import torch
import transformers

print('=== GPU ===')
try:
	print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], text=True))
except FileNotFoundError:
	print('nvidia-smi not found — CPU-only environment, model load will OOM')

print('=== Disk (cwd) ===')
print(subprocess.check_output(['df', '-h', '.'], text=True))

print('=== Deps ===')
print(f'torch={torch.__version__}, cuda={torch.version.cuda}, transformers={transformers.__version__}')
print(f'cuda available: {torch.cuda.is_available()}, devices: {torch.cuda.device_count()}')

# Llama-3.1-8B bf16 ≈ 16 GB weights; need ~20 GB free with activations + KV cache headroom
if torch.cuda.is_available():
	free_gb = torch.cuda.mem_get_info()[0] / 1e9
	print(f'free VRAM: {free_gb:.1f} GB')
	if free_gb < 20:
		print('warning: <20 GB free VRAM. Llama-8B bf16 may OOM during generation.')


## Cell 1.6 — confirm HF Write access to the artifact repo

Critical when someone other than the repo owner runs this notebook (e.g. the over-18 teammate who accepted the Llama license is running on Brayden's SageMaker). If their HF token doesn't have Write on `BraydenF/desperation-circuit-artifacts`, the final upload cells will 403 and the run's results are stuck on the instance. This cell probes that BEFORE the model load.

In [ ]:
# whose HF token is active, and does it have Write access to the artifact repo?
# fail fast — discovering the 403 after 8 GPU-hours is a budget-killer. This probe
# uploads a tiny placeholder then immediately deletes it, so nothing ends up visible
# in the repo's file tree. (Commits remain in history — that's how HF works.)
import os
import tempfile
from huggingface_hub import whoami, upload_file, delete_file

from src.lib.config import load_config
cfg = load_config()
REPO_ID = cfg['paths']['hf_artifact_repo']

who = whoami()
HF_USER = who['name']
# expose to later cells so HF upload commit messages can attribute the run
os.environ['HF_USER'] = HF_USER
print(f'HF identity: {HF_USER}')
print(f'target repo: {REPO_ID}')

# write-access probe: upload then delete
_probe_in_repo = '.write_access_probe'
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as _f:
	_f.write('write-access probe (auto-deleted)\n')
	_probe_path = _f.name
try:
	upload_file(
		path_or_fileobj=_probe_path,
		path_in_repo=_probe_in_repo,
		repo_id=REPO_ID,
		repo_type='dataset',
		commit_message='probe: write-access check',
	)
except Exception as e:
	raise RuntimeError(
		f'\nHF Write to {REPO_ID} FAILED for user {HF_USER}:\n  {e}\n\n'
		f'Fix: the repo owner needs to add you as a Write collaborator at\n'
		f'  https://huggingface.co/datasets/{REPO_ID}/settings\n'
		f'Owner navigates to Settings -> Collaborators -> Add user -> {HF_USER} -> Write.\n'
		f'Stop the notebook here; do NOT burn GPU-hours until this is fixed.'
	)

# clean up so the probe doesn't show in the file tree of a public repo
try:
	delete_file(
		path_in_repo=_probe_in_repo,
		repo_id=REPO_ID,
		repo_type='dataset',
		commit_message='probe: cleanup',
	)
	print(f'OK — Write access confirmed; probe file deleted from repo tree.')
except Exception as e:
	print(f'warning: probe upload OK but delete failed ({e}); the placeholder file may be visible at HEAD until manually removed')


## Cell 2 — confirm stories on disk

Stories are shared with the Gemma run (`data/stories/{emotion}/*.txt`, committed in-tree). This cell just sanity-checks one per emotion.

In [ ]:
import sys
sys.path.insert(0, '.')
from pathlib import Path
from src.lib.config import load_config

cfg = load_config()
data_dir = Path(cfg['paths']['data_dir'])

for emotion in cfg['extraction']['emotions'] + ['neutral']:
	sample = (data_dir / 'stories' / emotion / '000.txt').read_text(encoding='utf-8')
	print(f'\n=== {emotion.upper()} (story 000, first 400 chars) ===')
	print(sample[:400])


## Cell 3 — load Llama-3.1-8B-Instruct

In [ ]:
from src.lib.model_load import load_llama

model, tokenizer = load_llama()
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, d_model={model.config.hidden_size}')


## Cell 4 — run extraction

In [ ]:
from src.extract_vectors import extract_all

# model_key='llama' routes outputs to outputs/m1_vectors/llama_L{layer}/ and reads
# extraction_layer from cfg['models']['llama']
results = extract_all(model, tokenizer, model_key='llama')
for emotion, r in results.items():
    print(f'{emotion}: vector shape={r.vector.shape}, n_stories_used={r.n_stories_used}')


## Cell 5 — sanity check the vectors

Cosine matrix between emotion vectors. Same gate as Gemma: if desperation's cosine with any control exceeds ~0.5, the controls aren't clean and we re-examine story generation before moving to M2. If the cosines look noisy *only* on Llama this is also the trigger to revisit the layer-21 default in config.yaml.

In [ ]:
import json
import numpy as np
import pandas as pd

from src.lib.config import layer_suffix
lsuf = layer_suffix(cfg, 'llama')
log = json.loads((Path(cfg['paths']['outputs_dir']) / 'm1_vectors' / lsuf / 'extraction_log.json').read_text())
print('extraction log:')
print(json.dumps(log, indent=2))

emotions = cfg['extraction']['emotions']
vecs = {e: results[e].vector for e in emotions}
cos = pd.DataFrame(
    {e1: {e2: float(vecs[e1] @ vecs[e2]) for e2 in emotions} for e1 in emotions}
)
print('\npairwise cosines:')
cos


## Outputs

- `outputs/m1_vectors/llama_L{layer}/{emotion}.npy` — ℓ₂-normalized Llama emotion vectors
- `outputs/m1_vectors/llama_L{layer}/extraction_log.json` — protocol params + pairwise cosines

## Push to HF Hub

In [ ]:
import os
from huggingface_hub import HfApi
from src.lib.config import layer_suffix

api = HfApi()
lsuf = layer_suffix(cfg, 'llama')
hf_user = os.environ.get('HF_USER', 'unknown')
api.upload_folder(
    folder_path=f'outputs/m1_vectors/{lsuf}',
    repo_id=cfg['paths']['hf_artifact_repo'],
    repo_type='dataset',
    path_in_repo=f'm1_vectors/{lsuf}',
    commit_message=f'M1 Llama-3.1-8B-Instruct: emotion vectors (extracted by HF user {hf_user})',
)


## Next

Once cosines look clean, run `m2_steer_mmlu_llama.ipynb` (steering + MMLU capability gate).